# DETR3D in Pure PyTorch: A Literate Architecture

This output-free notebook builds the faithful DETR3D architecture locally, one tensor contract at a time. Architecture code does not import the production model; its immutable reference is production commit `5244860688cfc390a82f8c730017853188b50e5f`. Reduced synthetic probes are the default, while real data, checkpoint, training, and official evaluation paths are explicit opt-ins.

The faithful image contract is important: camera images are resized to `900x1600`, then padded only along height to `928x1600` before entering the model. Projection geometry remains tied to the unpadded `900x1600` image.

```text
six resized images [B,6,3,900,1600]
  -> official padding [B,6,3,928,1600]
  |
  +-> GridMask during training
  |
  +-> shared Caffe-style ResNet-101 + modulated DCNv2
  |     C3 [B,6,512,116,200]
  |     C4 [B,6,1024,58,100]
  |     C5 [B,6,2048,29,50]
  |
  +-> FPN
  |     P3 [B,6,256,116,200]
  |     P4 [B,6,256,58,100]
  |     P5 [B,6,256,29,50]
  |     P6 [B,6,256,15,25]
  |
900 query/content + query-position embeddings [B,900,256]
  -> initial normalized 3D references [B,900,3]
  -> decoder layer x6
       self-attention
       project references into N cameras
       sample and fuse L pyramid levels
       feed-forward network
       class [B,900,10], encoded box [B,900,10]
       detach refined references for the next layer
  -> all class logits [6,B,900,10]
  -> all encoded boxes [6,B,900,10]
```

## Symbols And Execution Modes

| Symbol | Meaning | Faithful value |
|---|---|---:|
| `B` | batch size | training dependent |
| `N` | synchronized camera views | 6 |
| `Q` | object queries | 900 |
| `C` | decoder/FPN channels | 256 |
| `L` | FPN levels | 4 |
| `K` | nuScenes classes | 10 |
| `D` | encoded box coordinates | 10 |

`SMOKE_MODE` keeps probes deterministic and small. Every expensive or asset-dependent path has a separate switch and is disabled unless requested.

In [ ]:
from __future__ import annotations

import random
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import models
from torchvision.models import ResNet101_Weights
from torchvision.ops import DeformConv2d

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

SMOKE_MODE = True
RUN_REAL_DATA = False
RUN_CHECKPOINT = False
RUN_TRAIN_STEP = True
RUN_OFFICIAL_EVAL = False
NUSCENES_DATAROOT = None
CHECKPOINT_PATH = None
FCOS3D_CHECKPOINT_PATH = None

ARCH-GRID-MASK

## GridMask: Shared Structured Occlusion

GridMask draws one periodic binary canvas and broadcasts it across every flattened camera image and color channel. For grid spacing `d`, ratio `r`, and random phase `(s_h, s_w)`, the removed stripe width is

```text
l = clip(round(d * r), 1, d - 1)
horizontal stripe i: [i*d + s_h, i*d + s_h + l)
vertical stripe   j: [j*d + s_w, j*d + s_w + l)
```

A `1.5H x 1.5W` canvas permits rotation before a centered `H x W` crop. Production flattens cameras first, `[B,N,3,H,W] -> [B*N,3,H,W]`; the mask `[1,1,H,W]` therefore broadcasts over `B*N` and RGB. Applying this module directly to a five-dimensional probe has the equivalent broadcast `[1,1,1,H,W]`, so all views still share the sampled pattern. In `mode=1`, the mask is inverted. Evaluation mode and the probability gate return the original tensor unchanged.

In [ ]:
class GridMask(nn.Module):
    def __init__(
        self,
        use_h: bool = True,
        use_w: bool = True,
        rotate: int = 1,
        offset: bool = False,
        ratio: float = 0.5,
        mode: int = 1,
        probability: float = 0.7,
    ):
        super().__init__()
        self.use_h = use_h
        self.use_w = use_w
        self.rotate = rotate
        self.offset = offset
        self.ratio = ratio
        self.mode = mode
        self.probability = probability

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        if not self.training or np.random.rand() > self.probability:
            return images
        height, width = images.shape[-2:]
        canvas_height = int(1.5 * height)
        canvas_width = int(1.5 * width)
        spacing = np.random.randint(2, height)
        line_width = min(max(int(spacing * self.ratio + 0.5), 1), spacing - 1)
        mask = np.ones((canvas_height, canvas_width), dtype=np.float32)  # symbolic canvas [1.5H,1.5W] -> mask [1.5H,1.5W]; faithful [1392,2400] -> [1392,2400]
        start_h = np.random.randint(spacing)
        start_w = np.random.randint(spacing)
        if self.use_h:
            for index in range(canvas_height // spacing):
                start = spacing * index + start_h
                mask[start : min(start + line_width, canvas_height), :] = 0
        if self.use_w:
            for index in range(canvas_width // spacing):
                start = spacing * index + start_w
                mask[:, start : min(start + line_width, canvas_width)] = 0

        rotation = np.random.randint(self.rotate) if self.rotate > 1 else 0
        mask = np.asarray(Image.fromarray(mask).rotate(rotation)).copy()
        top = (canvas_height - height) // 2
        left = (canvas_width - width) // 2
        mask = mask[top : top + height, left : left + width]  # symbolic [1.5H,1.5W] -> [H,W]; faithful [1392,2400] -> [928,1600]
        mask_tensor = torch.as_tensor(mask, device=images.device, dtype=images.dtype)
        if self.mode == 1:
            mask_tensor = 1 - mask_tensor
        mask_tensor = mask_tensor.view(1, 1, height, width)  # symbolic [H,W] -> [1,1,H,W]; faithful [928,1600] -> [1,1,928,1600]
        if not self.offset:
            return images * mask_tensor  # symbolic [BN,3,H,W] * [1,1,H,W] -> [BN,3,H,W]; faithful [B*6,3,928,1600] * [1,1,928,1600] -> [B*6,3,928,1600]

        noise = torch.as_tensor(
            2 * (np.random.rand(height, width) - 0.5),
            device=images.device,
            dtype=images.dtype,
        ).view(1, 1, height, width)  # symbolic [H,W] -> [1,1,H,W]; faithful [928,1600] -> [1,1,928,1600]
        return images * mask_tensor + noise * (1 - mask_tensor)  # symbolic [BN,3,H,W] with [1,1,H,W] -> [BN,3,H,W]; faithful [B*6,3,928,1600] with [1,1,928,1600] -> [B*6,3,928,1600]


In [ ]:
grid_probe = GridMask(probability=1.0)
grid_images = torch.ones(2, 3, 3, 16, 24)
grid_probe.eval()
grid_eval = grid_probe(grid_images)
assert grid_eval is grid_images
grid_probe.train()
grid_train = grid_probe(grid_images)
assert grid_train.shape == (2, 3, 3, 16, 24)
assert torch.equal(grid_train[:, 0], grid_train[:, 1])

ARCH-BACKBONE

## Caffe ResNet-101 And Modulated DCNv2

Each ResNet bottleneck normally samples a fixed `k x k` grid. DCNv2 predicts an `(x,y)` displacement and a sigmoid modulation value for each of the `k^2` locations:

```text
conv_offset(x) = concat(offset, mask_logits) [B*N,3*k^2,H_o,W_o]
offset                                  [B*N,2*k^2,H_o,W_o]
mask = sigmoid(mask_logits)             [B*N,  k^2,H_o,W_o]
deform_conv(x, offset, mask)            [B*N,C_o,H_o,W_o]
```

Zero-initialized offset/mask prediction starts with zero displacement and modulation `sigmoid(0)=0.5`, matching the pinned production module. The original dense convolution weights are copied into `deform_conv`. The offset-only `DeformConv2dPack` is retained because it is the production fallback when `official_style=False`; the faithful path selects the modulated pack.

Official Caffe stride placement moves each downsampling stride from bottleneck `conv2` to `conv1`. Stage 4 and stage 5 then replace every `conv2` with modulated DCNv2. Cameras share one backbone by flattening `B` and `N`, then restoring the camera axis:

```text
images [B,N,3,H,W] -> [B*N,3,H,W] -> shared ResNet -> restore B,N
faithful input [B,6,3,928,1600]
  C3 [B,6, 512,116,200]
  C4 [B,6,1024, 58,100]
  C5 [B,6,2048, 29, 50]
```

The stem is frozen for `frozen_stages >= 0`. Calling `train()` re-applies freezing, and `norm_eval=True` keeps every BatchNorm layer in evaluation mode so running statistics do not drift.

In [ ]:
class DeformConv2dPack(nn.Module):
    """Offset-only deformable convolution used by the non-official fallback."""

    def __init__(self, conv: nn.Conv2d):
        super().__init__()
        kernel_h, kernel_w = conv.kernel_size
        offset_channels = 2 * kernel_h * kernel_w
        self.offset_conv = nn.Conv2d(
            in_channels=conv.in_channels,
            out_channels=offset_channels,
            kernel_size=conv.kernel_size,
            stride=conv.stride,
            padding=conv.padding,
            dilation=conv.dilation,
            groups=1,
            bias=True,
        )
        self.deform_conv = DeformConv2d(
            in_channels=conv.in_channels,
            out_channels=conv.out_channels,
            kernel_size=kernel_h,
            stride=conv.stride[0],
            padding=conv.padding[0],
            dilation=conv.dilation[0],
            groups=conv.groups,
            bias=conv.bias is not None,
        )
        with torch.no_grad():
            self.deform_conv.weight.copy_(conv.weight)
            if conv.bias is not None and self.deform_conv.bias is not None:
                self.deform_conv.bias.copy_(conv.bias)
            nn.init.zeros_(self.offset_conv.weight)
            nn.init.zeros_(self.offset_conv.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        offset = self.offset_conv(x)  # symbolic [BN,C_i,H_i,W_i] -> [BN,2k^2,H_o,W_o]; faithful-sized fallback k=3 stage4 first [B*6,256,116,200] -> [B*6,18,58,100], stage5 first [B*6,512,58,100] -> [B*6,18,29,50]
        return self.deform_conv(x, offset)  # symbolic [BN,C_i,H_i,W_i] -> [BN,C_o,H_o,W_o]; faithful-sized fallback stage4 first [B*6,256,116,200] -> [B*6,256,58,100], stage5 first [B*6,512,58,100] -> [B*6,512,29,50]


class ModulatedDeformConv2dPack(nn.Module):
    def __init__(self, conv: nn.Conv2d):
        super().__init__()
        kernel_h, kernel_w = conv.kernel_size
        self.offset_channels = 2 * kernel_h * kernel_w
        self.mask_channels = kernel_h * kernel_w
        self.conv_offset = nn.Conv2d(
            in_channels=conv.in_channels,
            out_channels=self.offset_channels + self.mask_channels,
            kernel_size=conv.kernel_size,
            stride=conv.stride,
            padding=conv.padding,
            dilation=conv.dilation,
            groups=1,
            bias=True,
        )
        self.deform_conv = DeformConv2d(
            in_channels=conv.in_channels,
            out_channels=conv.out_channels,
            kernel_size=conv.kernel_size,
            stride=conv.stride,
            padding=conv.padding,
            dilation=conv.dilation,
            groups=conv.groups,
            bias=conv.bias is not None,
        )
        with torch.no_grad():
            self.deform_conv.weight.copy_(conv.weight)
            if conv.bias is not None and self.deform_conv.bias is not None:
                self.deform_conv.bias.copy_(conv.bias)
            nn.init.zeros_(self.conv_offset.weight)
            nn.init.zeros_(self.conv_offset.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        offset_mask = self.conv_offset(x)  # symbolic [BN,C_i,H_i,W_i] -> [BN,3k^2,H_o,W_o]; faithful k=3 stage4 [B*6,256,58,100] -> [B*6,27,58,100], stage5 [B*6,512,29,50] -> [B*6,27,29,50]
        offset = offset_mask[:, : self.offset_channels]  # symbolic [BN,3k^2,H_o,W_o] -> [BN,2k^2,H_o,W_o]; faithful stage4 [B*6,27,58,100] -> [B*6,18,58,100], stage5 [B*6,27,29,50] -> [B*6,18,29,50]
        mask = offset_mask[:, self.offset_channels :].sigmoid()  # symbolic [BN,3k^2,H_o,W_o] -> [BN,k^2,H_o,W_o]; faithful stage4 [B*6,27,58,100] -> [B*6,9,58,100], stage5 [B*6,27,29,50] -> [B*6,9,29,50]
        return self.deform_conv(x, offset, mask)  # symbolic [BN,C_i,H_i,W_i] -> [BN,C_o,H_o,W_o]; faithful stage4 [B*6,256,58,100] -> [B*6,256,58,100], stage5 [B*6,512,29,50] -> [B*6,512,29,50]


In [ ]:
class MultiViewImageBackbone(nn.Module):
    """Apply one torchvision ResNet-101 to every camera view."""

    def __init__(
        self,
        variant: str = "resnet101",
        pretrained: bool = True,
        frozen_stages: int = 1,
        norm_eval: bool = True,
        official_style: bool = False,
    ):
        super().__init__()
        if variant != "resnet101":
            raise ValueError(f"Unsupported backbone variant: {variant}")

        weights = ResNet101_Weights.IMAGENET1K_V2 if pretrained else None
        backbone = models.resnet101(weights=weights)
        if official_style:
            self._use_caffe_stride(backbone)
        self.stem = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
        )
        self.stage2 = backbone.layer1
        self.stage3 = backbone.layer2
        self.stage4 = backbone.layer3
        self.stage5 = backbone.layer4
        self._convert_stage_to_deformable(self.stage4, modulated=official_style)
        self._convert_stage_to_deformable(self.stage5, modulated=official_style)
        self.frozen_stages = frozen_stages
        self.norm_eval = norm_eval
        self._freeze_stages()

    @staticmethod
    def _use_caffe_stride(backbone: nn.Module) -> None:
        for stage in (backbone.layer2, backbone.layer3, backbone.layer4):
            block = stage[0]
            block.conv1.stride = (2, 2)
            block.conv2.stride = (1, 1)

    @staticmethod
    def _convert_stage_to_deformable(stage: nn.Sequential, *, modulated: bool) -> None:
        pack_type = ModulatedDeformConv2dPack if modulated else DeformConv2dPack
        for block in stage:
            if hasattr(block, "conv2") and isinstance(block.conv2, nn.Conv2d):
                block.conv2 = pack_type(block.conv2)

    def _freeze_stages(self) -> None:
        if self.frozen_stages >= 0:
            self.stem.eval()
            for param in self.stem.parameters():
                param.requires_grad = False
        stages = [self.stage2, self.stage3, self.stage4, self.stage5]
        for idx, stage in enumerate(stages, start=1):
            if self.frozen_stages >= idx:
                stage.eval()
                for param in stage.parameters():
                    param.requires_grad = False

    def train(self, mode: bool = True):
        super().train(mode)
        self._freeze_stages()
        if mode and self.norm_eval:
            for module in self.modules():
                if isinstance(module, nn.BatchNorm2d):
                    module.eval()
        return self

    def forward(self, images: torch.Tensor) -> Dict[str, torch.Tensor]:
        batch, num_cams, channels, height, width = images.shape
        flat = images.reshape(batch * num_cams, channels, height, width)  # symbolic [B,N,3,H,W] -> [BN,3,H,W]; faithful [B,6,3,928,1600] -> [B*6,3,928,1600]

        x = self.stem(flat)  # symbolic [BN,3,H,W] -> [BN,64,H/4,W/4]; faithful [B*6,3,928,1600] -> [B*6,64,232,400]
        stage2 = self.stage2(x)  # symbolic [BN,64,H/4,W/4] -> [BN,256,H/4,W/4]; faithful [B*6,64,232,400] -> [B*6,256,232,400]
        stage3 = self.stage3(stage2)  # symbolic [BN,256,H/4,W/4] -> [BN,512,H/8,W/8]; faithful [B*6,256,232,400] -> [B*6,512,116,200]
        stage4 = self.stage4(stage3)  # symbolic [BN,512,H/8,W/8] -> [BN,1024,H/16,W/16]; faithful [B*6,512,116,200] -> [B*6,1024,58,100]
        stage5 = self.stage5(stage4)  # symbolic [BN,1024,H/16,W/16] -> [BN,2048,H/32,W/32]; faithful [B*6,1024,58,100] -> [B*6,2048,29,50]

        features = {"stage3": stage3, "stage4": stage4, "stage5": stage5}
        multi_view = {}
        for name, feat in features.items():
            _, out_channels, out_height, out_width = feat.shape
            multi_view[name] = feat.reshape(batch, num_cams, out_channels, out_height, out_width)  # symbolic [BN,C_l,H_l,W_l] -> [B,N,C_l,H_l,W_l]; faithful C3 [B*6,512,116,200] -> [B,6,512,116,200], C4 [B*6,1024,58,100] -> [B,6,1024,58,100], C5 [B*6,2048,29,50] -> [B,6,2048,29,50]
        return multi_view


ARCH-FPN

## ImageFPN: A Four-Level Top-Down Pyramid

The feature pyramid first projects all backbone stages to `C=256`, then carries coarse semantic information toward finer maps:

```text
L_l = Conv1x1(C_l)
T_l = L_l + nearest_upsample(T_(l+1))
P_l = Conv3x3(T_l)
P6  = Conv3x3_stride2(ReLU(P5))

C5 ---- 1x1 ---- T5 ---- 3x3 ---- P5 ---- ReLU ---- stride-2 ---- P6
                   | upsample
C4 ---- 1x1 ---- (+) ---- 3x3 ---- P4
                   | upsample
C3 ---- 1x1 ---- (+) ---- 3x3 ---- P3
```

The pinned class exposes `relu_before_extra_convs`; faithful construction sets it to `True`. For padded `928x1600` inputs the restored multi-view outputs are `P3 [B,6,256,116,200]`, `P4 [B,6,256,58,100]`, `P5 [B,6,256,29,50]`, and `P6 [B,6,256,15,25]`.

In [ ]:
class ImageFPN(nn.Module):
    def __init__(
        self,
        in_channels: Iterable[int] = (512, 1024, 2048),
        out_channels: int = 256,
        out_names: Iterable[str] = ("p3", "p4", "p5", "p6"),
        relu_before_extra_convs: bool = False,
    ):
        super().__init__()
        self.out_names = list(out_names)
        self.relu_before_extra_convs = relu_before_extra_convs
        self.lateral_convs = nn.ModuleList(
            [nn.Conv2d(ch, out_channels, kernel_size=1) for ch in in_channels]
        )
        self.output_convs = nn.ModuleList(
            [nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1) for _ in in_channels]
        )
        self.extra_conv = nn.Conv2d(
            out_channels, out_channels, kernel_size=3, stride=2, padding=1
        )

    def forward(self, features: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        stage_names = sorted(features.keys())
        flat_feats = []
        batch, num_cams = next(iter(features.values())).shape[:2]
        for name in stage_names:
            feat = features[name]
            flat = feat.reshape(batch * num_cams, feat.shape[2], feat.shape[3], feat.shape[4])  # symbolic [B,N,C_l,H_l,W_l] -> [BN,C_l,H_l,W_l]; faithful C3 [B,6,512,116,200] -> [B*6,512,116,200], C4 [B,6,1024,58,100] -> [B*6,1024,58,100], C5 [B,6,2048,29,50] -> [B*6,2048,29,50]
            flat_feats.append(flat)

        laterals = []
        for conv, feat in zip(self.lateral_convs, flat_feats):
            lateral = conv(feat)  # symbolic [BN,C_l,H_l,W_l] -> [BN,256,H_l,W_l]; faithful C3 [B*6,512,116,200] -> [B*6,256,116,200], C4 [B*6,1024,58,100] -> [B*6,256,58,100], C5 [B*6,2048,29,50] -> [B*6,256,29,50]
            laterals.append(lateral)
        for idx in range(len(laterals) - 1, 0, -1):
            upsampled = F.interpolate(laterals[idx], size=laterals[idx - 1].shape[-2:], mode="nearest")  # symbolic [BN,256,H_(l+1),W_(l+1)] -> [BN,256,H_l,W_l]; faithful [B*6,256,29,50] -> [B*6,256,58,100] or [B*6,256,58,100] -> [B*6,256,116,200]
            laterals[idx - 1] = laterals[idx - 1] + upsampled  # [B*N,256,H_l,W_l]

        outputs = []
        for conv, feat in zip(self.output_convs, laterals):
            output = conv(feat)  # symbolic [BN,256,H_l,W_l] -> [BN,256,H_l,W_l]; faithful P3 [B*6,256,116,200] -> [B*6,256,116,200], P4 [B*6,256,58,100] -> [B*6,256,58,100], P5 [B*6,256,29,50] -> [B*6,256,29,50]
            outputs.append(output)
        extra_input = F.relu(outputs[-1]) if self.relu_before_extra_convs else outputs[-1]
        outputs.append(self.extra_conv(extra_input))  # symbolic [BN,256,H_5,W_5] -> [BN,256,ceil(H_5/2),ceil(W_5/2)]; faithful [B*6,256,29,50] -> [B*6,256,15,25]

        pyramid: Dict[str, torch.Tensor] = {}
        for out_name, feat in zip(self.out_names, outputs):
            pyramid[out_name] = feat.reshape(batch, num_cams, feat.shape[1], feat.shape[2], feat.shape[3])  # symbolic [BN,256,H_l,W_l] -> [B,N,256,H_l,W_l]; faithful P3 [B*6,256,116,200] -> [B,6,256,116,200], P4 [B*6,256,58,100] -> [B,6,256,58,100], P5 [B*6,256,29,50] -> [B,6,256,29,50], P6 [B*6,256,15,25] -> [B,6,256,15,25]
        return pyramid


## Reduced Deterministic Image-Path Probe

A single `64x96` camera image preserves the real ResNet-101/DCNv2/FPN channel contracts while reducing spatial work. Evaluation mode disables stochastic and BatchNorm updates. The expected backbone sizes are `8x12`, `4x6`, and `2x3`; stride-2 P6 is therefore `1x2`.

In [ ]:
if SMOKE_MODE:
    smoke_backbone = MultiViewImageBackbone(
        pretrained=False, official_style=True
    ).eval()
    smoke_fpn = ImageFPN(relu_before_extra_convs=True).eval()
    smoke_images = torch.randn(1, 1, 3, 64, 96)
    with torch.inference_mode():
        smoke_features = smoke_backbone(smoke_images)  # symbolic [B,N,3,H,W] -> C3/C4/C5 [B,N,C_l,H_l,W_l]; faithful [B,6,3,928,1600] -> C3 [B,6,512,116,200], C4 [B,6,1024,58,100], C5 [B,6,2048,29,50]; smoke [1,1,3,64,96] -> [1,1,512,8,12]/[1,1,1024,4,6]/[1,1,2048,2,3]
        smoke_pyramid = smoke_fpn(smoke_features)  # symbolic C3/C4/C5 [B,N,C_l,H_l,W_l] -> P3/P4/P5/P6 [B,N,256,H_l,W_l]; faithful C3/C4/C5 -> P3 [B,6,256,116,200], P4 [B,6,256,58,100], P5 [B,6,256,29,50], P6 [B,6,256,15,25]; smoke -> [1,1,256,8,12]/[1,1,256,4,6]/[1,1,256,2,3]/[1,1,256,1,2]

    assert list(smoke_features) == ["stage3", "stage4", "stage5"]
    assert [tensor.ndim for tensor in smoke_features.values()] == [5, 5, 5]
    assert [tensor.shape[:2] for tensor in smoke_features.values()] == [(1, 1)] * 3
    assert [tensor.shape[-2:] for tensor in smoke_features.values()] == [(8, 12), (4, 6), (2, 3)]
    assert list(smoke_pyramid) == ["p3", "p4", "p5", "p6"]
    assert all(tensor.ndim == 5 for tensor in smoke_pyramid.values())
    assert all(tensor.shape[:3] == (1, 1, 256) for tensor in smoke_pyramid.values())
    pyramid_sizes = [tensor.shape[-2:] for tensor in smoke_pyramid.values()]
    assert pyramid_sizes == [(8, 12), (4, 6), (2, 3), (1, 2)]
    assert all(
        fine_h > coarse_h and fine_w > coarse_w
        for (fine_h, fine_w), (coarse_h, coarse_w) in zip(
            pyramid_sizes, pyramid_sizes[1:]
        )
    )
    assert smoke_pyramid["p6"].shape == (1, 1, 256, 1, 2)

ARCH-REFERENCES

## Reference Points: Normalized Parameters To Metric Geometry

The decoder keeps each query center as a bounded normalized reference `r = (r_x,r_y,r_z)` in `[0,1]^3`. Projection needs the corresponding metric point `p` in the LiDAR frame. For component-wise point-cloud bounds `p_min` and `p_max`,

```text
p = p_min + r * (p_max - p_min)
normalized reference [B,Q,3] -> metric LiDAR point [B,Q,3]
```

The rank and dimensions do not change, but the coordinate space does. Iterative refinement later works in unconstrained logit space, `inverse_sigmoid(r) = log(r / (1-r))`; clamping to `[eps,1-eps]` keeps endpoints finite.

In [ ]:
def inverse_sigmoid(x: torch.Tensor, eps: float = 1e-5) -> torch.Tensor:
    x = x.clamp(min=eps, max=1 - eps)
    return torch.log(x / (1 - x))


def denormalize_reference_points(reference_points: torch.Tensor, pc_range) -> torch.Tensor:
    pc_range_tensor = torch.as_tensor(pc_range, dtype=reference_points.dtype, device=reference_points.device)
    xyz_min = pc_range_tensor[:3]
    xyz_max = pc_range_tensor[3:]
    return reference_points * (xyz_max - xyz_min) + xyz_min  # symbolic normalized [B,Q,3] -> metric LiDAR [B,Q,3]; faithful normalized [B,900,3] -> metric LiDAR [B,900,3]


ARCH-FEATURE-SAMPLING

## Projecting Queries And Sampling Every Camera Level

Append a homogeneous coordinate to metric point `p_q`, project it with camera matrix `P_n`, and divide by positive depth:

```text
p_h(q)       = [p_x, p_y, p_z, 1]^T
s(n,q)       = P_n p_h(q) = [s_x, s_y, d, s_w]^T
u(n,q)       = s_x / max(d, eps)
v(n,q)       = s_y / max(d, eps)
g_x(n,q)     = 2*u/W_n - 1
g_y(n,q)     = 2*v/H_n - 1
valid(n,q)   = (d > eps) and (-1 <= g_x <= 1) and (-1 <= g_y <= 1)
```

`W_n` and `H_n` are the resized, unpadded image dimensions from metadata. The resulting grid follows `grid_sample(..., align_corners=False)`, exactly as production does. Invalid points may still be sampled as zeros, but the explicit mask is the authoritative rule used by attention. Geometry is computed in float32; sampled values return to the FPN feature dtype.

```text
normalized refs [B,Q,3]
        | denormalize + append 1
        v
homogeneous [B,1,Q,4,1] -- P [B,N,1,4,4] --> pixels [B,N,Q,2]
                                                        | normalize
                                                        v
FPN level [B,N,C,H_l,W_l] -> [B*N,C,H_l,W_l]   grid [B*N,Q,1,2]
                              \______________________/
                                      grid_sample
                                           |
                                           v
                         restore/permute [B,C,Q,N,1]
                                           | stack L
                                           v
                              sampled [B,C,Q,N,1,L]
                              mask    [B,1,Q,N,1,L]
```

In [ ]:
def _stack_projection_matrices(
    img_metas: List[Dict], key: str, device: torch.device, dtype: torch.dtype
) -> torch.Tensor:
    matrices = []
    for meta in img_metas:
        if key not in meta:
            raise KeyError(f"Expected '{key}' in img_metas for DETR3D feature sampling.")
        matrices.append(torch.as_tensor(meta[key], dtype=dtype, device=device))
    return torch.stack(matrices, dim=0)  # symbolic B x [N,4,4] -> [B,N,4,4]; faithful B x [6,4,4] -> [B,6,4,4]


def _stack_image_shapes(
    img_metas: List[Dict], num_cams: int, device: torch.device, dtype: torch.dtype
) -> torch.Tensor:
    shapes = []
    for meta in img_metas:
        image_shape = meta.get("image_shape")
        if image_shape is None:
            image_shape = meta.get("img_shape")
        if image_shape is None:
            raise KeyError("Expected 'image_shape' or 'img_shape' in img_metas.")
        shape_tensor = torch.as_tensor(image_shape, dtype=dtype, device=device)
        if shape_tensor.ndim == 1:
            shape_tensor = shape_tensor.unsqueeze(0).repeat(num_cams, 1)  # symbolic shared [2] -> per-camera [N,2]; faithful shared [2] -> [6,2]
        shapes.append(shape_tensor[:, :2])  # symbolic camera shape [N,S] -> height-width [N,2]; faithful [6,S] -> [6,2]
    return torch.stack(shapes, dim=0)  # symbolic B x [N,2] -> [B,N,2]; faithful B x [6,2] -> [B,6,2]


def feature_sampling(
    mlvl_feats: List[torch.Tensor],
    reference_points: torch.Tensor,
    pc_range,
    img_metas: List[Dict],
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    if not img_metas:
        raise ValueError("img_metas is required for DETR3D feature sampling.")

    batch, num_queries, _ = reference_points.shape
    _, num_cams, channels, _, _ = mlvl_feats[0].shape
    num_levels = len(mlvl_feats)
    device = reference_points.device
    feat_dtype = mlvl_feats[0].dtype
    geom_dtype = torch.float32

    ref_points_3d = denormalize_reference_points(reference_points.to(dtype=geom_dtype), pc_range)  # symbolic normalized [B,Q,3] -> metric LiDAR [B,Q,3]; faithful normalized [B,900,3] -> metric LiDAR [B,900,3]
    ref_points_homo = torch.cat([ref_points_3d, torch.ones_like(ref_points_3d[..., :1])], dim=-1)  # symbolic metric [B,Q,3] + ones [B,Q,1] -> homogeneous [B,Q,4]; faithful [B,900,3] -> [B,900,4]

    matrix_key = "lidar2img" if "lidar2img" in img_metas[0] else "ego2img"
    proj = _stack_projection_matrices(img_metas, matrix_key, device, geom_dtype)
    if proj.ndim == 3:
        proj = proj.unsqueeze(1).repeat(1, num_cams, 1, 1)  # symbolic shared [B,4,4] -> per-camera [B,N,4,4]; faithful shared [B,4,4] -> [B,6,4,4]
    image_shapes = _stack_image_shapes(img_metas, num_cams, device, geom_dtype)

    points = ref_points_homo[:, None, :, :, None]  # symbolic [B,Q,4] -> [B,1,Q,4,1]; faithful [B,900,4] -> [B,1,900,4,1]
    proj_points = torch.matmul(proj[:, :, None], points).squeeze(-1)  # symbolic [B,N,1,4,4] @ [B,1,Q,4,1] -> [B,N,Q,4,1] -> [B,N,Q,4]; faithful [B,6,1,4,4] @ [B,1,900,4,1] -> [B,6,900,4,1] -> [B,6,900,4]

    depth = proj_points[..., 2:3]  # symbolic projected [B,N,Q,4] -> depth [B,N,Q,1]; faithful [B,6,900,4] -> [B,6,900,1]
    xy = proj_points[..., :2] / depth.clamp(min=1e-5)  # symbolic projected [B,N,Q,4] -> pixels [B,N,Q,2]; faithful [B,6,900,4] -> [B,6,900,2]
    width = image_shapes[..., 1:2]  # symbolic image shapes [B,N,2] -> width [B,N,1]; faithful [B,6,2] -> [B,6,1]
    height = image_shapes[..., 0:1]  # symbolic image shapes [B,N,2] -> height [B,N,1]; faithful [B,6,2] -> [B,6,1]
    norm_x = xy[..., 0] / width.clamp(min=1.0)  # symbolic [B,N,Q,2] -> [B,N,Q]; faithful [B,6,900,2] -> [B,6,900]
    norm_y = xy[..., 1] / height.clamp(min=1.0)  # symbolic [B,N,Q,2] -> [B,N,Q]; faithful [B,6,900,2] -> [B,6,900]
    grid = torch.stack([norm_x * 2 - 1, norm_y * 2 - 1], dim=-1)  # symbolic [B,N,Q] + [B,N,Q] -> [B,N,Q,2]; faithful [B,6,900] + [B,6,900] -> [B,6,900,2]

    valid = depth[..., 0] > 1e-5  # symbolic depth [B,N,Q,1] -> validity [B,N,Q]; faithful [B,6,900,1] -> [B,6,900]
    valid = valid & (grid[..., 0] >= -1.0) & (grid[..., 0] <= 1.0) & (grid[..., 1] >= -1.0) & (grid[..., 1] <= 1.0)  # symbolic grid [B,N,Q,2] -> x/y tests [B,N,Q] -> validity [B,N,Q]; faithful [B,6,900,2] -> [B,6,900]

    sampled_levels = []
    masks = []
    for level_feat in mlvl_feats:
        feat = level_feat.reshape(batch * num_cams, channels, level_feat.shape[-2], level_feat.shape[-1]).float()  # symbolic [B,N,C,H_l,W_l] -> [BN,C,H_l,W_l]; faithful [B,6,256,H_l,W_l] -> [B*6,256,H_l,W_l]
        level_grid = grid.reshape(batch * num_cams, num_queries, 1, 2)  # symbolic [B,N,Q,2] -> [BN,Q,1,2]; faithful [B,6,900,2] -> [B*6,900,1,2]
        sampled = F.grid_sample(feat, level_grid, mode="bilinear", padding_mode="zeros", align_corners=False)  # symbolic [BN,C,H_l,W_l] with [BN,Q,1,2] -> [BN,C,Q,1]; faithful [B*6,256,H_l,W_l] -> [B*6,256,900,1]
        sampled = sampled.reshape(batch, num_cams, channels, num_queries, 1).permute(0, 2, 3, 1, 4).to(feat_dtype)  # symbolic [BN,C,Q,1] -> [B,N,C,Q,1] -> [B,C,Q,N,1]; faithful [B*6,256,900,1] -> [B,256,900,6,1]
        level_mask = valid.permute(0, 2, 1)[:, None, :, :, None].to(dtype=feat_dtype)  # symbolic [B,N,Q] -> [B,Q,N] -> [B,1,Q,N,1]; faithful [B,6,900] -> [B,1,900,6,1]
        sampled_levels.append(sampled)
        masks.append(level_mask)

    sampled_feats = torch.stack(sampled_levels, dim=-1)  # symbolic L x [B,C,Q,N,1] -> [B,C,Q,N,1,L]; faithful 4 x [B,256,900,6,1] -> [B,256,900,6,1,4]
    mask = torch.stack(masks, dim=-1)  # symbolic L x [B,1,Q,N,1] -> [B,1,Q,N,1,L]; faithful 4 x [B,1,900,6,1] -> [B,1,900,6,1,4]
    return ref_points_3d, sampled_feats, mask


## Analytic Projection Probe

An identity-like camera maps `(x,y,z)` to pixel `(x/z,y/z)`. The `4x4` feature below stores `10*y+x` at each pixel center. The visible metric point `(2,2,1)` maps to normalized grid `(0,0)`, halfway among values `11,12,21,22`, so bilinear sampling must return `16.5`. The other points isolate the out-of-frame and negative-depth mask branches.

In [ ]:
geometry_feature = torch.tensor([[[[[0.0, 1.0, 2.0, 3.0], [10.0, 11.0, 12.0, 13.0], [20.0, 21.0, 22.0, 23.0], [30.0, 31.0, 32.0, 33.0]]]]])
geometry_references = torch.tensor([[[0.2, 0.2, 0.75], [0.5, 0.2, 0.75], [0.1, 0.1, 0.25]]])
geometry_metas = [{"lidar2img": torch.eye(4)[None], "image_shape": (4, 4)}]  # symbolic one [4,4] matrix -> [N=1,4,4]; probe [4,4] -> [1,4,4]
geometry_metric, geometry_sampled, geometry_mask = feature_sampling(
    [geometry_feature],
    geometry_references,
    (0.0, 0.0, -2.0, 10.0, 10.0, 2.0),
    geometry_metas,
)
torch.testing.assert_close(geometry_metric, torch.tensor([[[2.0, 2.0, 1.0], [5.0, 2.0, 1.0], [1.0, 1.0, -1.0]]]))
assert geometry_sampled.shape == (1, 1, 3, 1, 1, 1)
torch.testing.assert_close(geometry_sampled[0, 0, :, 0, 0, 0], torch.tensor([16.5, 0.0, 0.0]))  # symbolic [B,C,Q,N,1,L] -> [Q]; probe [1,1,3,1,1,1] -> [3]
assert geometry_mask.shape == (1, 1, 3, 1, 1, 1)
torch.testing.assert_close(geometry_mask[0, 0, :, 0, 0, 0], torch.tensor([1.0, 0.0, 0.0]))  # symbolic [B,1,Q,N,1,L] -> [Q]; probe [1,1,3,1,1,1] -> [3]


ARCH-CROSS-ATTENTION

## Learned Camera-Level Cross-Attention

Each query predicts one independent sigmoid gate for every camera and pyramid level. This is not a softmax: several views or levels may contribute together. For sampled channel `c`, query `q`, camera `n`, and level `l`,

```text
a(b,q,n,l) = sigmoid(Linear(q + q_pos))[b,q,n,l]
f(b,c,q)   = sum_n sum_l sampled(b,c,q,n,1,l) * a(b,q,n,l) * mask(b,1,q,n,1,l)
out         = query + dropout(output_proj(f)) + position_encoder(inverse_sigmoid(reference))
```

The attention linear layer starts at zero, so every valid camera-level gate starts at `sigmoid(0)=0.5`. The visual branch alone passes through dropout; there is exactly one query residual. Reference logits pass through a two-layer normalized position encoder before addition.

```text
query (+ query_pos) [B,Q,C] -> gates [B,1,Q,N,1,L]
                                      |
sampled [B,C,Q,N,1,L] x mask --------(x)
                                      | sum cameras + levels
                                      v
                                visual [B,Q,C] -> projection -> dropout
query [B,Q,C] -------------------------------(+)
reference [B,Q,3] -> inverse sigmoid -> position encoder --(+) -> [B,Q,C]
```

In [ ]:
class Detr3DCrossAttention(nn.Module):
    def __init__(
        self,
        embed_dims: int = 256,
        num_cams: int = 6,
        num_levels: int = 4,
        pc_range=(-51.2, -51.2, -5.0, 51.2, 51.2, 3.0),
        dropout: float = 0.1,
    ):
        super().__init__()
        self.embed_dims = embed_dims
        self.num_cams = num_cams
        self.num_levels = num_levels
        self.pc_range = pc_range
        self.dropout = nn.Dropout(dropout)
        self.attention_weights = nn.Linear(embed_dims, num_cams * num_levels)
        self.output_proj = nn.Linear(embed_dims, embed_dims)
        self.position_encoder = nn.Sequential(
            nn.Linear(3, embed_dims),
            nn.LayerNorm(embed_dims),
            nn.ReLU(inplace=True),
            nn.Linear(embed_dims, embed_dims),
            nn.LayerNorm(embed_dims),
            nn.ReLU(inplace=True),
        )
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.zeros_(self.attention_weights.weight)
        nn.init.zeros_(self.attention_weights.bias)
        nn.init.xavier_uniform_(self.output_proj.weight)
        nn.init.zeros_(self.output_proj.bias)

    def forward(
        self,
        query: torch.Tensor,
        mlvl_feats: List[torch.Tensor],
        reference_points: torch.Tensor,
        img_metas: List[Dict],
        query_pos: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        _, sampled_feats, mask = feature_sampling(
            mlvl_feats=mlvl_feats,
            reference_points=reference_points,
            pc_range=self.pc_range,
            img_metas=img_metas,
        )

        weight_query = query if query_pos is None else query + query_pos
        attention_weights = self.attention_weights(weight_query)  # symbolic [B,Q,C] -> [B,Q,N*L]; faithful [B,900,256] -> [B,900,24]
        attention_weights = attention_weights.view(query.shape[0], query.shape[1], self.num_cams, 1, self.num_levels)  # symbolic [B,Q,N*L] -> [B,Q,N,1,L]; faithful [B,900,24] -> [B,900,6,1,4]
        attention_weights = attention_weights.permute(0, 3, 1, 2, 4).unsqueeze(4)  # symbolic [B,Q,N,1,L] -> [B,1,Q,N,L] -> [B,1,Q,N,1,L]; faithful [B,900,6,1,4] -> [B,1,900,6,1,4]
        attention_weights = attention_weights.sigmoid().to(sampled_feats.dtype) * mask

        fused = (sampled_feats * attention_weights).sum(dim=-1).sum(dim=-1).sum(dim=-1)  # symbolic [B,C,Q,N,1,L] -> [B,C,Q]; faithful [B,256,900,6,1,4] -> [B,256,900]
        fused = fused.permute(0, 2, 1)  # symbolic [B,C,Q] -> [B,Q,C]; faithful [B,256,900] -> [B,900,256]
        fused = self.output_proj(fused)  # symbolic [B,Q,C] -> [B,Q,C]; faithful [B,900,256] -> [B,900,256]
        pos_feat = self.position_encoder(inverse_sigmoid(reference_points))  # symbolic reference [B,Q,3] -> position [B,Q,C]; faithful [B,900,3] -> [B,900,256]
        return self.dropout(fused) + query + pos_feat


## Reduced Cross-Attention Probe

A one-camera, one-level probe checks the output contract and initialization without requiring the full image path. Zero attention logits verify the production reset rule before the sampler, projection, residual, and position branches run together.

In [ ]:
attention_probe = Detr3DCrossAttention(
    embed_dims=1,
    num_cams=1,
    num_levels=1,
    pc_range=(0.0, 0.0, -2.0, 10.0, 10.0, 2.0),
    dropout=0.0,
).eval()
assert torch.count_nonzero(attention_probe.attention_weights.weight) == 0
assert torch.count_nonzero(attention_probe.attention_weights.bias) == 0
attention_query = torch.zeros(1, 1, 1)
attention_reference = torch.tensor([[[0.2, 0.2, 0.75]]])
with torch.inference_mode():
    attention_output = attention_probe(
        attention_query, [geometry_feature], attention_reference, geometry_metas
    )  # symbolic query [B,Q,C] -> attention output [B,Q,C]; faithful [B,900,256] -> [B,900,256]; probe [1,1,1] -> [1,1,1]
assert attention_output.shape == (1, 1, 1)
